# Research Question 5: User Adoption Patterns

## 🎯 Research Question  
**"Who adopts coding agents (newcomers vs experienced developers)?"**

## 📋 Methodology
- **User Classification**: Identify newcomers vs experienced developers
- **Adoption Metrics**:
  - First-time vs repeat AI agent usage
  - User activity patterns over time
  - Agent preference by user experience level
  - Temporal adoption trends

## 🔍 Expected Insights
- Understand user demographics for AI agent adoption
- Identify patterns in how different user types engage with AI tools
- Track adoption trends and user retention
- Establish user experience correlations with AI usage

## 📊 Classification Strategy
- **Newcomers**: Users with recent account creation or low activity
- **Experienced**: Users with extensive contribution history
- **Metrics**: PR count, account age (when available), activity patterns

## 🎯 Success Metrics
- User classification accuracy
- Adoption rate by user type
- Temporal trends in user adoption
- Agent preference patterns by experience level

In [1]:
# Setup for User Adoption Analysis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
from datetime import datetime, timedelta
from collections import Counter

# Add src directory to path
sys.path.append('../src')
from data_loader import load_aidev

print("📈 RQ5: User Adoption Patterns Analysis")
print("=" * 50)
print(f"📅 Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Load sample data
print("\n📂 Loading sample data...")
df = load_aidev(sample_size=10000)  # Larger sample for user patterns
print(f"✅ Loaded {len(df):,} PRs for user adoption analysis")

# Basic user statistics
def analyze_user_patterns(df):
    """Analyze basic user adoption patterns"""
    user_stats = {}
    
    # User activity summary
    user_activity = df.groupby('user').agg({
        'id': 'count',
        'agent': lambda x: list(x.unique()),
        'created_at': ['min', 'max'],
        'state': lambda x: (x == 'closed').sum() / len(x)  # Success rate
    }).round(3)
    
    user_activity.columns = ['total_prs', 'agents_used', 'first_pr', 'last_pr', 'success_rate']
    user_activity['agents_count'] = user_activity['agents_used'].apply(len)
    user_activity['days_active'] = (pd.to_datetime(user_activity['last_pr']) - 
                                   pd.to_datetime(user_activity['first_pr'])).dt.days
    
    # User classification (basic heuristic)
    # Newcomers: <= 5 PRs or <= 30 days active
    # Experienced: > 20 PRs and > 90 days active  
    # Regular: everything else
    def classify_user(row):
        if row['total_prs'] <= 5 or row['days_active'] <= 30:
            return 'newcomer'
        elif row['total_prs'] > 20 and row['days_active'] > 90:
            return 'experienced'
        else:
            return 'regular'
    
    user_activity['user_type'] = user_activity.apply(classify_user, axis=1)
    
    user_stats = {
        'total_users': len(user_activity),
        'user_types': user_activity['user_type'].value_counts().to_dict(),
        'avg_prs_per_user': user_activity['total_prs'].mean(),
        'median_prs_per_user': user_activity['total_prs'].median()
    }
    
    return user_activity, user_stats

# Convert created_at to datetime for analysis
try:
    df['created_at'] = pd.to_datetime(df['created_at'])
    datetime_available = True
except:
    print("⚠️ DateTime conversion failed - using basic analysis")
    datetime_available = False

user_activity, user_stats = analyze_user_patterns(df)

print(f"\n👥 USER STATISTICS:")
print(f"  Total Users: {user_stats['total_users']:,}")
print(f"  Average PRs per User: {user_stats['avg_prs_per_user']:.1f}")
print(f"  Median PRs per User: {user_stats['median_prs_per_user']:.1f}")

print(f"\n📊 USER TYPE DISTRIBUTION:")
for user_type, count in user_stats['user_types'].items():
    percentage = (count / user_stats['total_users']) * 100
    print(f"  {user_type.title()}: {count:,} users ({percentage:.1f}%)")

print(f"\n🔍 DateTime Analysis: {'✅ Available' if datetime_available else '❌ Limited'}")

📈 RQ5: User Adoption Patterns Analysis
📅 Analysis Date: 2025-10-12 03:52:34

📂 Loading sample data...
Loading dataset from local file: ../data/raw/aidata.csv


Loaded sample of 10000 rows
✅ Loaded 10,000 PRs for user adoption analysis

👥 USER STATISTICS:
  Total Users: 1,699
  Average PRs per User: 5.9
  Median PRs per User: 1.0

📊 USER TYPE DISTRIBUTION:
  Newcomer: 1,643 users (96.7%)
  Regular: 55 users (3.2%)
  Experienced: 1 users (0.1%)

🔍 DateTime Analysis: ✅ Available
